# MLnode Transformer Training Notebook

Train a **TransformerEncoder** classifier on MNIST — defined entirely as a MLnode JSON graph.

Each 28×28 image is treated as a **sequence of 28 rows** (28 features per step). The graph:

```
Input (B, 28, 28)
  → Linear 28→64            # embed each row into d_model
  → TransformerEncoder      # 2 layers, 4 heads, d_model=64 (built from flat params)
  → Flatten                 # (B, 28·64) = (B, 1792)
  → Linear 1792→10          # class logits
```

The `TransformerEncoder` node takes the flat `TransformerEncoderLayer` params plus
`num_layers` and `norm` — the backend composes the stack for you
(see `mlnode/layers/registry.py`). `batch_first: true` keeps everything in
the `(B, Seq, D)` convention that the shape validator assumes.

In [ ]:
# Install MLnode (run once)
# !pip install -e /path/to/mlnode
# Or if already installed:
from mlnode import MLnodeModel
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 1. Load the Model Graph

In [ ]:
model_json = "mnist_transformer.mlnode.json"
model = MLnodeModel.load(model_json)
print(model)

## 2. Validate Shapes Statically

No GPU memory is allocated — the validator propagates symbolic shapes
through the DAG, including through the transformer stack.

In [ ]:
result = model.validate()

print(f"Valid: {result.valid}")
print(f"\nShape propagation through the graph:")
for node_id, shape in result.node_shapes.items():
    print(f"  {node_id:15s} → {shape}")

if not result.valid:
    print(f"\nErrors:")
    for err in result.errors:
        print(f"  {err.message}")

## 3. Build the PyTorch Module

In [ ]:
module = model.build().to(device)

# Verify with a dummy input: a batch of 28-step sequences
dummy = torch.randn(1, 28, 28, device=device)
out = module(dummy)
print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {out.shape}")
print(f"Parameters:   {sum(p.numel() for p in module.parameters()):,}")
print(f"\nModule structure:")
print(module)

## 4. Data Pipeline

MNIST loaders. The dataset yields `(B, 1, 28, 28)`; we squeeze the channel
dimension in the training loop so each image becomes a `(28, 28)` sequence.

> **Note on positions:** this demo feeds the encoder without positional
> encodings, so it sees the rows as an unordered set plus whatever order
> information leaks through the residual stream. MNIST rows are distinctive
> enough that accuracy stays high; for sequence tasks where order matters,
> add a positional-encoding `custom` node to the graph.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST mean and std
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples:     {len(test_dataset):,}")

images, labels = next(iter(train_loader))
print(f"\nBatch from loader: {images.shape} → model input: {images.squeeze(1).shape}")

## 5. Training Loop

In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-3

optimizer = optim.AdamW(module.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"Optimizer:     AdamW (lr={LEARNING_RATE})")
print(f"Loss:          CrossEntropyLoss")
print(f"Epochs:        {EPOCHS}")
print(f"LR Scheduler:  CosineAnnealingLR")

In [ ]:
def train_epoch(module, loader, optimizer, criterion, device, epoch):
    module.train()
    total_loss, correct, seen = 0.0, 0, 0
    for batch_idx, (images, labels) in enumerate(loader):
        # (B, 1, 28, 28) → (B, 28, 28): rows as a sequence
        images = images.squeeze(1).to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = module(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        seen += labels.size(0)
        if batch_idx % 100 == 0:
            print(f"  epoch {epoch} [{seen:>6}/{len(loader.dataset)}] loss={loss.item():.4f}")
    return total_loss / seen, correct / seen


@torch.no_grad()
def evaluate(module, loader, criterion, device):
    module.eval()
    total_loss, correct, seen = 0.0, 0, 0
    for images, labels in loader:
        images = images.squeeze(1).to(device)
        labels = labels.to(device)
        logits = module(images)
        total_loss += criterion(logits, labels).item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        seen += labels.size(0)
    return total_loss / seen, correct / seen

In [ ]:
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_epoch(module, train_loader, optimizer, criterion, device, epoch)
    test_loss, test_acc = evaluate(module, test_loader, criterion, device)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)
    print(f"Epoch {epoch}: train acc {train_acc:.4f} | test acc {test_acc:.4f} "
          f"| test loss {test_loss:.4f} | {time.time()-t0:.1f}s")

## 6. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
epochs = range(1, len(history["train_loss"]) + 1)
axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["test_loss"], label="test")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(epochs, history["train_acc"], label="train")
axes[1].plot(epochs, history["test_acc"], label="test")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Sample Predictions

In [ ]:
module.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = module(images.squeeze(1).to(device)).argmax(1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, img, pred, true in zip(axes.flat, images, preds, labels):
    ax.imshow(img.squeeze(), cmap="gray")
    ok = "✓" if pred == true else "✗"
    ax.set_title(f"{pred.item()} {ok}", color="green" if pred == true else "red", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Save the Model

In [ ]:
# The architecture is already portable JSON — save the learned weights alongside it.
torch.save(module.state_dict(), "mnist_transformer_weights.pth")
model.save("mnist_transformer_copy.mlnode.json")
print("Saved weights and architecture JSON.")
print("Load later with: MLnodeModel.load(...).build().load_state_dict(torch.load(...))")